# 03 · Global / Local feature experiment (Phase 5)

Đây là notebook độc lập cho một Kaggle kernel mới. Notebook nhận cache Phase 4 làm input, tự clone repo/cài dependency, kiểm tra lại thứ tự `image_id`, rồi chạy cùng một MLP protocol cho ba representation:

- Global-only: CLS feature;
- Local-only: mean pooling patch tokens;
- Global + Local: nối hai vector 768 chiều.

Classifier protocol được giữ cố định theo head MLP đã chọn ở Phase 3: seed 42, hidden size 256, 20 epoch, learning rate `1e-3`, weight decay `1e-4`, threshold `0.5`. Không có source metadata nên kết quả chỉ là in-domain validation.

## Trước khi chạy

1. Chọn GPU trong Kaggle Notebook Settings.
2. Add Input là ZIP/cache đã tạo bởi `02_dinov3_feature_extraction.ipynb`, hoặc đặt `CACHE_ROOT` thủ công ở cell 3.
3. Đảm bảo repo trên GitHub có `src/input_data/feature_dataset.py` và `src/training/feature_trainer.py`.

Artifact được lưu vào `/kaggle/working/phase5_local_outputs/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_REF = 'main'
REPO_DIR = Path('/kaggle/working/FaceTrace')
attached_candidates = [Path('/kaggle/input/facetrace'), Path('/kaggle/input/face-trace')]
attached_repo = next((p for p in attached_candidates if (p / 'src').is_dir()), None)
if attached_repo is not None:
    REPO_DIR = attached_repo
elif not (REPO_DIR / 'src').is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        REPO_DIR = Path('/kaggle/working/FaceTrace_repo')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
required_paths = [REPO_DIR / 'src/models/classifier.py', REPO_DIR / 'src/input_data/feature_dataset.py', REPO_DIR / 'src/training/feature_trainer.py']
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Repo snapshot is missing required files: {missing}')
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)

In [ ]:
%pip install -q 'PyYAML>=6' 'pandas>=2' 'scikit-learn>=1.4' 'tqdm>=4.66'

In [ ]:
import json
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError('GPU chưa được bật. Chọn Settings → Accelerator → GPU rồi chạy lại.')

SEED = 43
DEVICE = torch.device('cuda')
CACHE_ROOT = None  # Set to a mounted cache directory if auto-discovery does not find it.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
# Locate either an unpacked Phase 4 cache or its ZIP in Kaggle Inputs.
if CACHE_ROOT is None:
    input_root = Path('/kaggle/input')
    manifest_candidates = sorted(input_root.rglob('cache_manifest.json')) if input_root.exists() else []
    if manifest_candidates:
        CACHE_ROOT = manifest_candidates[0].parent
    else:
        zip_candidates = sorted(input_root.rglob('*.zip')) if input_root.exists() else []
        if not zip_candidates:
            raise FileNotFoundError('Không tìm thấy cache_manifest.json hoặc ZIP Phase 4 trong /kaggle/input. Add artifact Phase 4 làm Input hoặc đặt CACHE_ROOT.')
        cache_zip = next((p for p in zip_candidates if 'phase4' in p.name.lower()), zip_candidates[0])
        extracted_root = Path('/kaggle/working/phase4_cache_input')
        extracted_root.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(cache_zip), str(extracted_root))
        manifest_candidates = sorted(extracted_root.rglob('cache_manifest.json'))
        if not manifest_candidates:
            raise FileNotFoundError('ZIP không chứa cache_manifest.json')
        CACHE_ROOT = manifest_candidates[0].parent
else:
    CACHE_ROOT = Path(CACHE_ROOT)

CACHE_ROOT = CACHE_ROOT.resolve()
CACHE_MANIFEST_PATH = CACHE_ROOT / 'cache_manifest.json'
if not CACHE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(CACHE_MANIFEST_PATH)
print('Cache:', CACHE_ROOT)

In [ ]:
def load_tensor(path: Path) -> torch.Tensor:
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

cache_manifest = json.loads(CACHE_MANIFEST_PATH.read_text(encoding='utf-8'))
expected_model = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
assert cache_manifest['model_id'] == expected_model
global_features = load_tensor(CACHE_ROOT / 'global_features.pt')
patch_features = load_tensor(CACHE_ROOT / 'patch_features.pt')
labels = load_tensor(CACHE_ROOT / 'labels.pt').to(torch.long)
metadata = pd.read_csv(CACHE_ROOT / 'metadata.csv', keep_default_na=False)

assert tuple(global_features.shape) == tuple(cache_manifest['global_shape'])
assert tuple(patch_features.shape) == tuple(cache_manifest['patch_shape'])
assert len(metadata) == len(global_features) == len(patch_features) == len(labels)
assert metadata['image_id'].is_unique
assert np.array_equal(metadata['label'].to_numpy(dtype=int), labels.numpy())

# Confirm persisted cache order against the saved split manifests.
for split_name in ('train', 'val'):
    split_manifest = pd.read_csv(CACHE_ROOT / f'{split_name}.csv', keep_default_na=False)
    cached_rows = metadata[metadata['split'] == split_name]
    assert cached_rows['image_id'].tolist() == split_manifest['image_id'].tolist()
    assert cached_rows['label'].astype(int).tolist() == split_manifest['label'].astype(int).tolist()

print('Model revision:', cache_manifest['model_revision'])
print('Global:', tuple(global_features.shape), global_features.dtype)
print('Patches:', tuple(patch_features.shape), patch_features.dtype)
print('Rows by split:', metadata['split'].value_counts().to_dict())
print('Persisted cache order verified.')

In [ ]:
from src.models.local import pool_patch_features

LOCAL_POOLING = 'mean'
global_features = global_features.float().contiguous()
local_features = pool_patch_features(patch_features.float(), method=LOCAL_POOLING).contiguous()
if global_features.shape[1] != local_features.shape[1]:
    raise AssertionError('Global and pooled local dimensions differ unexpectedly')

representations = {
    'global_only': global_features,
    'local_only': local_features,
    'global_local': torch.cat([global_features, local_features], dim=1),
}
print({name: tuple(value.shape) for name, value in representations.items()})
print('Fusion: concat; branch normalization: disabled; local pooling:', LOCAL_POOLING)

In [ ]:
from torch.utils.data import DataLoader

from src.input_data.feature_dataset import CachedFeatureDataset
from src.models.classifier import MLPClassifier
from src.training.feature_trainer import fit_feature_probe, predict_feature_model
from src.utils.seed import set_seed

local_config = yaml.safe_load((REPO_DIR / 'configs' / 'local.yaml').read_text(encoding='utf-8'))
train_cfg = local_config['training']
model_cfg = local_config['model']
eval_cfg = local_config['evaluation']
BATCH_SIZE = int(local_config['data']['batch_size'])
EPOCHS = int(train_cfg['epochs'])
LEARNING_RATE = float(train_cfg['learning_rate'])
WEIGHT_DECAY = float(train_cfg['weight_decay'])
THRESHOLD = float(eval_cfg['threshold'])
HIDDEN_DIM = int(model_cfg['hidden_dim'])
DROPOUT = float(model_cfg['dropout'])
MIXED_PRECISION = bool(train_cfg.get('mixed_precision', True))

all_image_ids = metadata['image_id'].tolist()
train_indices = metadata.index[metadata['split'] == 'train'].to_numpy()
val_indices = metadata.index[metadata['split'] == 'val'].to_numpy()
OUTPUT_DIR = REPO_DIR / 'outputs' / 'phase5_local_seed43'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'config.yaml').write_text(yaml.safe_dump(local_config, sort_keys=False), encoding='utf-8')
(OUTPUT_DIR / 'cache_manifest.json').write_text(json.dumps(cache_manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

results = {}
prediction_frames = []
for representation_name, feature_tensor in representations.items():
    set_seed(SEED)
    train_dataset = CachedFeatureDataset(feature_tensor, labels, train_indices, all_image_ids)
    val_dataset = CachedFeatureDataset(feature_tensor, labels, val_indices, all_image_ids)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    classifier = MLPClassifier(
        input_dim=feature_tensor.shape[1],
        hidden_dim=HIDDEN_DIM,
        num_classes=int(model_cfg['num_classes']),
        dropout=DROPOUT,
    ).to(DEVICE)
    head_dir = OUTPUT_DIR / representation_name
    best_state, best_epoch, best_metrics, history = fit_feature_probe(
        model=classifier,
        train_loader=train_loader,
        validation_loader=val_loader,
        device=DEVICE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        threshold=THRESHOLD,
        mixed_precision=MIXED_PRECISION,
        history_path=head_dir / 'history.csv',
    )
    head_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        'classifier': best_state,
        'representation': representation_name,
        'local_pooling': LOCAL_POOLING,
        'classifier_name': 'mlp',
        'hidden_dim': HIDDEN_DIM,
        'dropout': DROPOUT,
        'threshold': THRESHOLD,
        'seed': SEED,
        'best_epoch': best_epoch,
        'validation_metrics': best_metrics,
    }, head_dir / 'best_classifier.pt')
    predictions = predict_feature_model(classifier, val_loader, DEVICE)
    prediction_frame = metadata.iloc[predictions['row_index'].numpy()].copy()
    prediction_frame['representation'] = representation_name
    prediction_frame['prob_real'] = predictions['probabilities'][:, 0].numpy()
    prediction_frame['prob_fake'] = predictions['probabilities'][:, 1].numpy()
    prediction_frame['prediction'] = (prediction_frame['prob_fake'] >= THRESHOLD).astype(int)
    prediction_frames.append(prediction_frame)
    results[representation_name] = {
        'best_epoch': best_epoch,
        'metrics': best_metrics,
        'checkpoint': str(head_dir / 'best_classifier.pt'),
    }

with (OUTPUT_DIR / 'results.json').open('w', encoding='utf-8') as handle:
    json.dump(results, handle, indent=2, ensure_ascii=False)
predictions_all = pd.concat(prediction_frames, ignore_index=True)
predictions_all.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)
summary = pd.DataFrame([{'representation': name, 'best_epoch': value['best_epoch'], **value['metrics']} for name, value in results.items()])
summary.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
display(summary.sort_values(['val_auroc', 'val_balanced_accuracy'], ascending=False))

In [ ]:
selected = max(results, key=lambda name: (results[name]['metrics']['val_auroc'], results[name]['metrics']['val_balanced_accuracy']))
selection = {
    'selected_representation_by_validation': selected,
    'selection_rule': 'max validation AUROC, then validation Balanced Accuracy',
    'warning': 'In-domain validation only; no unseen-source test is available.',
}
(OUTPUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
archive = shutil.make_archive(str(REPO_DIR / 'outputs' / 'phase5_local_experiment_artifacts_seed43'), 'zip', root_dir=OUTPUT_DIR)
print('Selected representation:', selected)
print('Output directory:', OUTPUT_DIR)
print('Artifact ZIP:', archive)